In [ ]:
#| default_exp nbskill

## Installing the skill

Install the notebook workflow instructions, then register reusable local repositories.

The repository also ships the instructions that teach an agent how to use these tools. This notebook installs the bundled `SKILL.md` and references into local Codex or Claude skills directories, and can register the nbskill MCP server in Cursor's `mcp.json` so the notebook-first workflow can travel with the package.

The skill files are packaging for agent behavior, not another implementation path. They point agents back to the same notebook-aware tools in this repository, so users get consistent reads, writes, execution, and review through Python or MCP.

```python
build_skill_from_readme("README.md", "nbskill/SKILL.md")
install_nbskill(dest="~/.codex/skills/jupyter-notebooks")
```

### Production contract

Skill installation is Python-first. It builds `SKILL.md` from the documented README section, installs only into the requested Codex or Claude skills directory, writes Cursor MCP configuration only when Cursor is requested, leaves MCP processes running, returns restart guidance when needed, and installs hooks only when explicitly requested.

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
from nbskill.nbskill import build_skill_from_readme as _example_build_skill_from_readme
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
root = demo_path("06_skill_example")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    readme.write_text("before\n<!-- nbskill-skill:start -->\n# Skill Body\nUse read_nb first.\n<!-- nbskill-skill:end -->\nafter\n", encoding="utf-8")
    _example_build_skill_from_readme(str(readme), str(out))
    print("built:", out.name)
    print(out.read_text(encoding="utf-8").splitlines()[-2])
finally:
    remove_demo_path(root)

Built nbs/data/06_skill_example/SKILL.md from nbs/data/06_skill_example/README.md
built: SKILL.md
# Skill Body


In [ ]:
#| export
import json, subprocess, tomllib
from importlib.resources import files
from importlib.util import find_spec
from pathlib import Path


from nbskill.foundation import api_return, install_nbdev_pre_commit_hooks

In [ ]:
#| export
_SKILL_START = "<!-- nbskill-skill:start -->"
_SKILL_END = "<!-- nbskill-skill:end -->"
_SKILL_FRONTMATTER = """---
name: jupyter-notebooks
description: Work notebook-first in nbdev projects with nbskill MCP tools for reading, writing, updating, and executing notebooks without raw JSON.
---"""

In [ ]:
#| export
def _marked_readme_section(text, start=_SKILL_START, end=_SKILL_END):
    start_count = text.count(start)
    end_count = text.count(end)
    if start_count != 1 or end_count != 1:
        raise ValueError(f"README must contain exactly one {start!r} and one {end!r} marker")
    if text.index(start) > text.index(end):
        raise ValueError(f"README marker {start!r} must appear before {end!r}")
    return text.split(start, 1)[1].split(end, 1)[0].strip()

In [ ]:
#| export
def _skill_frontmatter(out_path):
    path = Path(out_path)
    if path.exists():
        text = path.read_text(encoding="utf-8")
        if text.startswith("---\n"):
            parts = text.split("\n---", 1)
            if len(parts) == 2: return f"---{parts[0][3:]}\n---"
    return _SKILL_FRONTMATTER

In [ ]:
#| export
def build_skill_from_readme(
    readme_path: str = "README.md",  # README containing the marked skill section
    out_path: str = "nbskill/SKILL.md",  # Skill file to write
):
    "Build SKILL.md from the marked section of README.md."
    out = Path(out_path)
    body = _marked_readme_section(Path(readme_path).read_text(encoding="utf-8"))
    text = f"{_skill_frontmatter(out)}\n\n{body}\n"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(text, encoding="utf-8")
    print(f"Built {out} from {readme_path}")
    return api_return(out)

### Installing agent instructions

The installer resolves the target skills directory, copies the packaged `SKILL.md`, and includes reference files. That keeps the project documentation and the agent workflow in sync with the package version.

In [ ]:
#| exporti
def _nbskill_mcp_pids():
    proc = subprocess.run(["ps", "-eo", "pid=,ppid=,command="], text=True, capture_output=True)
    if proc.returncode != 0: return []
    pids = []
    for line in proc.stdout.splitlines():
        parts = line.strip().split(None, 2)
        if len(parts) != 3: continue
        try: pid, ppid = int(parts[0]), int(parts[1])
        except ValueError: continue
        command = parts[2]
        if "nbskill_mcp" in command or "nbskill.mcp" in command: pids.append({"pid": pid, "ppid": ppid, "command": command.strip()})
    return pids

In [ ]:
#| export
def nbskill_mcp_restart_notice(timeout=5.0, force=True, dry_run=False):
    "Return the reconnect instruction; stdio servers are started by the MCP client."
    return {
        "running": bool(_nbskill_mcp_pids()),
        "restarted": False,
        "processes": _nbskill_mcp_pids(),
        "message": "Reconnect the MCP client to start a fresh nbskill MCP server.",
    }

In [ ]:
#| export
def _nbskill_source_project():
    """Return the current checkout when running from the nbskill source tree."""
    for root in [Path.cwd(), *Path.cwd().parents]:
        pyproject = root / "pyproject.toml"
        if not pyproject.exists(): continue
        try: text = pyproject.read_text(encoding="utf-8")
        except OSError: continue
        if 'name = "nbskill"' in text and (root / "nbskill" / "mcp.py").exists():
            return root.resolve()
    return None

In [ ]:
#| export
def _cursor_mcp_path(workspace=None):
    root = Path(workspace).expanduser() if workspace else Path.home()
    return root / ".cursor" / "mcp.json"

In [ ]:
#| export
def _cursor_nbskill_server_config():
    return {"command": "nbskill_mcp"}

In [ ]:
#| exporti
def _install_cursor_mcp(workspace=None, overwrite=True):
    """Install nbskill's MCP server into Cursor's mcp.json."""
    path = _cursor_mcp_path(workspace)
    if path.exists():
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, dict): raise ValueError(f"Cursor MCP config must be a JSON object: {path}")
    else: data = {}
    servers = data.setdefault("mcpServers", {})
    if not isinstance(servers, dict): raise ValueError(f"Cursor MCP config mcpServers must be an object: {path}")
    expected = _cursor_nbskill_server_config()
    if servers.get("nbskill") == expected:
        return {"installed": False, "reason": "already-current", "path": path}
    if "nbskill" in servers and not overwrite: raise FileExistsError(path)
    servers["nbskill"] = expected
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {
        "installed": True,
        "path": path,
        "server": "nbskill",
        "workspace": str(Path(workspace).expanduser()) if workspace else None,
    }

### Codex MCP configuration

A Codex install writes a project-local configuration in the selected workspace and starts nbskill from that workspace.

In [ ]:
#| export
def _codex_mcp_path(workspace=None):
    root = Path(workspace or ".").expanduser()
    return root / ".codex" / "config.toml"

In [ ]:
#| export
def _codex_nbskill_config(workspace):
    root = Path(workspace).expanduser().resolve()
    lines = [
        "[mcp_servers.nbskill]", "enabled = true", "required = true",
        'command = "nbskill_mcp"', f"cwd = {json.dumps(str(root))}",
        "startup_timeout_sec = 60", "tool_timeout_sec = 180",
    ]
    lines += [
        "", "[mcp_servers.nbskill.tools.edit_notebook]", 'approval_mode = "approve"', "",
        "[mcp_servers.nbskill.tools.context]", 'approval_mode = "approve"',
    ]
    return chr(10).join(lines)

In [ ]:
#| exporti
def _without_codex_nbskill(text):
    """Remove only nbskill MCP tables from Codex TOML text."""
    kept, skipping = [], False
    for line in text.splitlines():
        stripped = line.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            table = stripped[1:-1].strip()
            skipping = table == "mcp_servers.nbskill" or table.startswith("mcp_servers.nbskill.")
        if not skipping: kept.append(line)
    return "\n".join(kept).rstrip()

In [ ]:
#| exporti
def _install_codex_mcp(workspace=None, overwrite=True):
    """Install or repair nbskill's MCP server in a project's Codex config."""
    path = _codex_mcp_path(workspace)
    text = path.read_text(encoding="utf-8") if path.exists() else ""
    data = tomllib.loads(text) if text else {}
    servers = data.get("mcp_servers", {})
    if not isinstance(servers, dict): raise ValueError(f"Codex MCP config must be a table: {path}")
    expected_text = _codex_nbskill_config(workspace or ".")
    expected = tomllib.loads(expected_text)["mcp_servers"]["nbskill"]
    if servers.get("nbskill") == expected:
        return {"installed": False, "reason": "already-current", "path": path}
    if "nbskill" in servers and not overwrite: raise FileExistsError(path)
    prefix = _without_codex_nbskill(text)
    updated = (prefix + "\n\n" if prefix else "") + expected_text + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(updated, encoding="utf-8")
    return {"installed": True, "path": path, "server": "nbskill", "workspace": str(Path(workspace or ".").expanduser())}

### Coexisting with aai-coding

`nbskill` remains a separate MCP runtime, while `aai-coding` supplies the persistent Python session and lifecycle orientation. Installation adds a small managed integration to an available `aai-coding` checkout. The integration routes notebook-owned sources to nbskill, preserves unrelated text, and becomes a no-op once every managed rule is present.

In [ ]:
#| exporti
_AAI_SKILL_RULE = "- Use `exhash` for ordinary edits such as code, tests, configuration, and prose. In an nbdev project, `nbs/**/*.ipynb` and their generated Python modules belong to the nbskill MCP workflow: call `healthcheck`, inspect with `context` or `filter_context`, search prior art with `reference`, edit with `edit_notebook`, and verify with a focused notebook check. Do not use `exhash` to mutate notebook-owned sources."
_AAI_NBDEV_LINE = "NBDEV_MSG = '**This is an nbdev project: notebooks in `nbs/` are the source of truth, and exported `.py` files are autogenerated.** Use nbskill MCP for notebook-owned work: call `healthcheck`, read with `context` or `filter_context`, search prior art with `reference`, edit only with `edit_notebook`, then verify with a focused nbskill check. The persistent Python session and `exhash` remain for ordinary Python, configuration, and prose, but must not mutate `nbs/**/*.ipynb` or their generated modules.'"
_AAI_ORIENTATION_LINES = [
    "    project = Path.cwd()",
    "    try: nbdev = any(line.startswith('[tool.nbdev]') for line in (project/'pyproject.toml').open())",
    "    except OSError: nbdev = False",
    "    if nbdev: message += '\\n\\n' + NBDEV_MSG",
]
_AAI_README_BLOCK = """<!-- nbskill-integration:start -->
## Notebook integration

`aai-coding` owns the persistent Python session, ordinary file editing, configuration, prose, and lifecycle orientation. `nbskill` owns `nbs/**/*.ipynb`, nbdev-generated modules, notebook execution, and notebook review through its MCP server and the `jupyter-notebooks` skill.
<!-- nbskill-integration:end -->"""
_AAI_SETUP_BLOCK = """<!-- nbskill-integration:start -->
## nbskill notebook support

Install `nbskill`, run `install_nbskill --target codex`, and restart Codex. The installer registers the nbskill MCP server, installs the `jupyter-notebooks` skill, and keeps this integration current. Acceptance requires `healthcheck` to succeed and notebook-owned work to use `context`, `edit_notebook`, and a focused nbskill verification check.
<!-- nbskill-integration:end -->"""

In [ ]:
#| exporti
def _write_if_changed(path, text):
    old = path.read_text(encoding="utf-8") if path.exists() else ""
    if old == text: return False
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return True

In [ ]:
#| exporti
def _ensure_marked_block(path, block):
    text = path.read_text(encoding="utf-8") if path.exists() else ""
    start, end = "<!-- nbskill-integration:start -->", "<!-- nbskill-integration:end -->"
    if block in text: return False
    if start in text and end in text:
        before, rest = text.split(start, 1)
        _, after = rest.split(end, 1)
        updated = before.rstrip() + "\n\n" + block + after
    else: updated = text.rstrip() + "\n\n" + block + "\n"
    return _write_if_changed(path, updated)

In [ ]:
#| exporti
def _ensure_aai_skill(path):
    text = path.read_text(encoding="utf-8")
    terms = ("nbskill MCP", "healthcheck", "edit_notebook", "Do not use `exhash`")
    if all(term in text for term in terms): return False
    lines = text.splitlines()
    matches = [i for i,line in enumerate(lines) if line.startswith("- Use `exhash`")]
    if len(matches) != 1: raise ValueError(f"Cannot locate the aai-coding exhash rule in {path}")
    lines[matches[0]] = _AAI_SKILL_RULE
    return _write_if_changed(path, "\n".join(lines) + "\n")

In [ ]:
#| exporti
def _ensure_aai_harness(path):
    text = path.read_text(encoding="utf-8")
    lines, changed = text.splitlines(), False
    terms = ("NBDEV_MSG", "nbskill MCP", "edit_notebook", "nbs/**/*.ipynb")
    if not all(term in text for term in terms):
        matches = [i for i,line in enumerate(lines) if line.startswith("NBDEV_MSG =")]
        if len(matches) == 1: lines[matches[0]] = _AAI_NBDEV_LINE
        else:
            anchors = [i for i,line in enumerate(lines) if line.startswith("BOOTSTRAP_MSG =")]
            if len(anchors) != 1: raise ValueError(f"Cannot locate the aai-coding bootstrap message in {path}")
            lines.insert(anchors[0] + 1, _AAI_NBDEV_LINE)
        changed = True
    text = "\n".join(lines) + "\n"
    if "if nbdev: message += '\\n\\n' + NBDEV_MSG" not in text:
        lines = text.splitlines()
        start = next((i for i,line in enumerate(lines) if line.startswith("def codex_orientation(")), None)
        anchor = next((i for i,line in enumerate(lines[start + 1:], start + 1) if line.startswith("    sample = ")), None) if start is not None else None
        if anchor is None: raise ValueError(f"Cannot locate codex_orientation sample setup in {path}")
        lines[anchor + 1:anchor + 1] = _AAI_ORIENTATION_LINES
        text, changed = "\n".join(lines) + "\n", True
    return _write_if_changed(path, text) if changed else False

In [ ]:
#| exporti
def _aai_coding_root(root=None):
    candidates = [Path(root).expanduser()] if root else []
    if not root:
        spec = find_spec("aai_coding")
        if spec and spec.origin: candidates.append(Path(spec.origin).parent.parent)
        source = _nbskill_source_project()
        if source: candidates.append(source.parent / "aai-coding")
        candidates += [
            Path.cwd(), Path.cwd().parent / "aai-coding",
            Path.home() / "Projects" / "aai-coding", Path.home() / "projects" / "aai-coding",
        ]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "aai_coding" / "harness.py").exists() and (candidate / "skills" / "persistent-python" / "SKILL.md").exists():
            return candidate.resolve()
    return None

In [ ]:
#| export
def install_aai_coding_integration(
    root: str | Path | None = None, # aai-coding checkout; auto-detected when omitted
):
    """Install nbskill routing into an aai-coding checkout without repeating completed edits."""
    root = _aai_coding_root(root)
    if root is None: return api_return({"installed": False, "changed": [], "reason": "aai-coding-not-found"})
    changed = []
    skill_path = root / "skills" / "persistent-python" / "SKILL.md"
    harness_path = root / "aai_coding" / "harness.py"
    readme_path, setup_path = root / "README.md", root / "SETUP.md"
    if _ensure_aai_skill(skill_path): changed.append(str(skill_path.relative_to(root)))
    if _ensure_aai_harness(harness_path): changed.append(str(harness_path.relative_to(root)))
    if _ensure_marked_block(readme_path, _AAI_README_BLOCK): changed.append(str(readme_path.relative_to(root)))
    if _ensure_marked_block(setup_path, _AAI_SETUP_BLOCK): changed.append(str(setup_path.relative_to(root)))
    return api_return({"installed": bool(changed), "changed": changed, "root": root})

`install_aai_coding_integration` updates only the managed notebook-routing fragments in an available `aai-coding` checkout. It reports the files it changed and leaves a current checkout untouched, which makes repeated nbskill installations safe.

In [ ]:
#| exporti
def _install_skill_tree(root, skill_name, skill_text, references, overwrite=True):
    """Copy changed skill files and leave byte-identical files untouched."""
    dst_dir = root / skill_name
    files_to_install = [(dst_dir / "SKILL.md", skill_text)]
    if references.is_dir():
        files_to_install += [
            (dst_dir / "references" / ref.name, ref.read_text(encoding="utf-8"))
            for ref in references.iterdir() if ref.is_file()
        ]
    changed, unchanged = [], []
    for path,text in files_to_install:
        if path.exists() and path.read_text(encoding="utf-8") != text and not overwrite:
            raise FileExistsError(path)
        if _write_if_changed(path, text): changed.append(path)
        else: unchanged.append(path)
    return changed, unchanged

In [ ]:
#| export
def install_nbskill(
    target: str = "codex",  # codex, claude, cursor, both, or custom when skills_dir is set
    skills_dir: str | None = None,  # Parent skills directory; skill is installed below jupyter-notebooks
    skill_name: str = "jupyter-notebooks",  # Skill folder name
    overwrite: bool = True,  # Update stale managed files and server entries
    install_hooks: bool = False,  # Install nbdev-clean/nbdev-test pre-commit hooks in the current repo
    restart_mcp: bool = True,  # Report whether the MCP client needs reconnecting
    cursor_workspace: str | None = None,  # Cursor workspace for .cursor/mcp.json; omit for global ~/.cursor/mcp.json
    codex_workspace: str | None = ".",  # Project workspace for .codex/config.toml
    reference_roots: str = "~/projects",  # Local Git repositories to add to the reference index
    index_references: bool = True,  # Index references during installation
    integrate_aai_coding: bool = True,  # Keep an available aai-coding checkout routed to nbskill
    aai_coding_dir: str | None = None,  # Explicit aai-coding checkout; auto-detected when omitted
):
    """Install current nbskill instructions, MCP configuration, and aai-coding routing."""
    target = target.lower()
    cursor_mcp = {"installed": False, "reason": "target-not-cursor"}
    codex_mcp = {"installed": False, "reason": "target-not-codex"}
    aai_coding = {"installed": False, "changed": [], "reason": "disabled-or-not-codex"}
    if target == "cursor": roots = []
    elif skills_dir: roots = [Path(skills_dir).expanduser()]
    elif target == "codex": roots = [Path.home() / ".codex" / "skills"]
    elif target in {"claude", "claude-code", "claude_code"}: roots = [Path.home() / ".claude" / "skills"]
    elif target == "both": roots = [Path.home() / ".codex" / "skills", Path.home() / ".claude" / "skills"]
    else: raise ValueError("target must be codex, claude, cursor, both, or use skills_dir")
    package = files("nbskill")
    skill_text = package.joinpath("SKILL.md").read_text(encoding="utf-8")
    references = package.joinpath("references")
    installed, unchanged = [], []
    for root in roots:
        changed, current = _install_skill_tree(root, skill_name, skill_text, references, overwrite)
        installed += changed
        unchanged += current
    if target == "cursor": cursor_mcp = _install_cursor_mcp(workspace=cursor_workspace, overwrite=overwrite)
    if not skills_dir and target in {"codex", "both"}:
        codex_mcp = _install_codex_mcp(workspace=codex_workspace, overwrite=overwrite)
        if integrate_aai_coding: aai_coding = install_aai_coding_integration(aai_coding_dir)
    hooks = install_nbdev_pre_commit_hooks(Path.cwd()) if install_hooks else {"installed": False, "reason": "disabled"}
    from nbskill.knowledge import reference_discover
    reference_index = reference_discover(roots=reference_roots, ingest=index_references)
    mcp_notice = {"running": False, "message": "not checked"}
    if restart_mcp and not skills_dir and target in {"codex", "both", "cursor"}:
        mcp_notice = nbskill_mcp_restart_notice()
    for path in installed: print(f"Installed {path}")
    if cursor_mcp.get("installed"): print(f"Installed Cursor MCP config at {cursor_mcp['path']}")
    if codex_mcp.get("installed"): print(f"Installed Codex MCP config at {codex_mcp['path']}")
    if aai_coding.get("installed"): print(f"Updated aai-coding integration at {aai_coding['root']}")
    if hooks.get("installed"): print(f"Installed nbdev pre-commit hook at {hooks['hook']}")
    if mcp_notice.get("running"):
        print("nbskill MCP restarted:" if mcp_notice.get("restarted") else "nbskill MCP restart incomplete:")
        print(mcp_notice["message"])
    return api_return({
        "installed": installed, "unchanged": unchanged, "cursor_mcp": cursor_mcp, "codex_mcp": codex_mcp,
        "aai_coding": aai_coding, "hooks": hooks, "references": reference_index, "mcp_restart": mcp_notice,
    })

In [ ]:
#| eval: false
install_nbskill(target="cursor", cursor_workspace=".")

In [ ]:
#| hide
from unittest.mock import patch

In [ ]:
#| hide
codex_install_root = demo_path("06_skill_codex_wrapper")
try:
    with patch.object(Path, "home", return_value=codex_install_root):
        with patch("nbskill.knowledge.reference_discover", return_value={}):
            result = install_nbskill(
                target="codex", codex_workspace=str(codex_install_root), restart_mcp=False,
                index_references=False, integrate_aai_coding=False,
            )
    server = tomllib.loads((codex_install_root / ".codex" / "config.toml").read_text())["mcp_servers"]["nbskill"]
    assert result["codex_mcp"]["installed"]
    assert server["cwd"] == str(codex_install_root.resolve())
    assert "args" not in server
    assert "args" not in tomllib.loads(_codex_nbskill_config(codex_install_root))["mcp_servers"]["nbskill"]
finally: remove_demo_path(codex_install_root)

In [ ]:
#| hide
install_root = demo_path("06_skill_install")
try:
    with patch("nbskill.knowledge.reference_discover", return_value={}):
        first = install_nbskill(skills_dir=str(install_root), index_references=False)
    skill_dir = install_root / "jupyter-notebooks"
    assert first["installed"]
    assert (skill_dir / "SKILL.md").exists()
    assert (skill_dir / "references" / "mcp-tools.md").exists()
    assert (skill_dir / "references" / "conversion.md").exists()
    assert (skill_dir / "references" / "extended-tools.md").exists()
    installed_text = {path: path.read_bytes() for path in skill_dir.rglob("*") if path.is_file()}
    with patch("nbskill.knowledge.reference_discover", return_value={}):
        second = install_nbskill(skills_dir=str(install_root), index_references=False)
    assert not second["installed"]
    assert second["unchanged"]
    assert installed_text == {path: path.read_bytes() for path in skill_dir.rglob("*") if path.is_file()}
finally:
    remove_demo_path(install_root)

In [ ]:
#| hide
cursor_root = demo_path("06_skill_cursor_install")
try:
    (cursor_root / ".cursor").mkdir(parents=True)
    existing = {"mcpServers": {"other": {"command": "python", "args": ["server.py"]}}}
    path = cursor_root / ".cursor" / "mcp.json"
    path.write_text(json.dumps(existing), encoding="utf-8")
    first = _install_cursor_mcp(cursor_root)
    installed_text = path.read_text(encoding="utf-8")
    second = _install_cursor_mcp(cursor_root)
    config = json.loads(installed_text)
    assert first["installed"] and not second["installed"]
    assert second["reason"] == "already-current" and path.read_text(encoding="utf-8") == installed_text
    assert config["mcpServers"]["other"]["command"] == "python"
    assert config["mcpServers"]["nbskill"] == {"command": "nbskill_mcp"}
    assert "args" not in _cursor_nbskill_server_config()
finally:
    remove_demo_path(cursor_root)

In [ ]:
#| hide
codex_root = demo_path("06_skill_codex_install")
try:
    config_path = codex_root / ".codex" / "config.toml"
    config_path.parent.mkdir(parents=True)
    config_path.write_text(
        "[mcp_servers.other]\ncommand = 'other_mcp'\n\n[mcp_servers.nbskill]\ncommand = 'stale'\n",
        encoding="utf-8",
    )
    first = _install_codex_mcp(codex_root)
    installed_text = config_path.read_text(encoding="utf-8")
    second = _install_codex_mcp(codex_root)
    config = tomllib.loads(installed_text)
    server = config["mcp_servers"]["nbskill"]
    assert first["installed"] and not second["installed"]
    assert second["reason"] == "already-current" and config_path.read_text(encoding="utf-8") == installed_text
    assert config["mcp_servers"]["other"]["command"] == "other_mcp"
    assert server["command"] == "nbskill_mcp" and server["cwd"] == str(codex_root.resolve())
    assert server["tools"]["edit_notebook"]["approval_mode"] == "approve"
    assert server["tools"]["context"]["approval_mode"] == "approve"
finally: remove_demo_path(codex_root)

In [ ]:
#| hide
root = demo_path("06_skill_build")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    readme.write_text(
        """before
<!-- nbskill-skill:start -->
# Skill Body
Use notebooks.
<!-- nbskill-skill:end -->
after
""",
        encoding="utf-8",
    )
    build_skill_from_readme(str(readme), str(out))
    text = out.read_text(encoding="utf-8")
    assert text.startswith("---\nname: jupyter-notebooks")
    assert "# Skill Body" in text
    assert "before" not in text and "after" not in text
    out.write_text("---\nname: custom-skill\ndescription: Keep me.\n---\nold\n", encoding="utf-8")
    build_skill_from_readme(str(readme), str(out))
    text = out.read_text(encoding="utf-8")
    assert text.startswith("---\nname: custom-skill")
    assert "# Skill Body" in text and "old" not in text
finally:
    remove_demo_path(root)


In [ ]:
#| hide
root = demo_path("06_skill_duplicate_markers")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    marked = "<!-- nbskill-skill:start -->\n# Body\n<!-- nbskill-skill:end -->\n"
    readme.write_text(marked + marked, encoding="utf-8")
    try: build_skill_from_readme(str(readme), str(out))
    except ValueError as exc: assert "exactly one" in str(exc)
    else: raise AssertionError("duplicate skill markers should fail")
finally:
    remove_demo_path(root)

In [ ]:
#| hide
import tomllib
from pathlib import Path


In [ ]:
#| hide
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
scripts = tomllib.loads((project_root / "pyproject.toml").read_text(encoding="utf-8"))["project"]["scripts"]
assert "batch_edit_nb" in scripts
assert "nbskill_status" in scripts
assert "install_nbskill" in scripts


In [ ]:
#| hide
for removed in ("nbskill-mcp", "private-symbol-report", "symbol-graph", "update-cell", "show-doc"):
    for path in [project_root / "README.md", project_root / "nbskill/SKILL.md", *(project_root / "nbskill/references").glob("*.md")]:
        if path.exists(): assert removed not in path.read_text(encoding="utf-8")


In [ ]:
#| hide
hook_root = demo_path("06_skill_hooks")
try:
    hook_root.mkdir()
    (hook_root / "nbs").mkdir()
    subprocess.run(["git", "init"], cwd=hook_root, check=True, capture_output=True)
    hook_result = install_nbdev_pre_commit_hooks(hook_root, run_nbdev_install_hooks=False)
    pre_commit = hook_root / ".git" / "hooks" / "pre-commit"
    hook_text = pre_commit.read_text(encoding="utf-8")
    assert hook_result["installed"]
    assert "nbdev-clean" in hook_text
    assert "nbdev-test" in hook_text
    assert (hook_root / ".git" / "info" / "nbskill-hooks-installed").exists()

    pre_commit.unlink()
    removed_result = install_nbdev_pre_commit_hooks(hook_root, run_nbdev_install_hooks=False)
    assert not removed_result["installed"]
    assert removed_result["reason"] == "hooks-removed-by-user"
    assert not pre_commit.exists()
    assert (hook_root / ".git" / "info" / "nbskill-hooks-disabled").exists()
finally:
    remove_demo_path(hook_root)


In [ ]:
#| hide
install_root = demo_path("06_skill_custom_install")
try:
    install_nbskill(
        target="custom", skills_dir=str(install_root), install_hooks=False, restart_mcp=False,
    )
    assert (install_root / "jupyter-notebooks" / "SKILL.md").exists()
    notice = nbskill_mcp_restart_notice(dry_run=True)
    assert "running" in notice and "message" in notice
finally:
    remove_demo_path(install_root)

In [ ]:
#| hide
aai_root = demo_path("06_skill_aai_coding")
try:
    (aai_root / "skills" / "persistent-python").mkdir(parents=True)
    (aai_root / "aai_coding").mkdir()
    (aai_root / "README.md").write_text("# aai-coding\n", encoding="utf-8")
    (aai_root / "SETUP.md").write_text("# Setup\n", encoding="utf-8")
    (aai_root / "skills" / "persistent-python" / "SKILL.md").write_text(
        "# Persistent Python\n\n- Use `exhash` for ALL edits -- code, tests, config, prose, notebook cells.\n", encoding="utf-8",
    )
    (aai_root / "aai_coding" / "harness.py").write_text(
        "BOOTSTRAP_MSG = 'bootstrap'\n"
        "NBDEV_MSG = 'Use doc(nbdev.skill) and edit notebook cells with exhash.'\n\n"
        "def codex_orientation(o):\n"
        "    message = 'orientation'\n"
        "    sample = 'sample'\n"
        "    message += sample\n", encoding="utf-8",
    )
    install_kw = dict(
        target="codex", codex_workspace=str(aai_root), restart_mcp=False, index_references=False,
        aai_coding_dir=str(aai_root),
    )
    with patch.object(Path, "home", return_value=aai_root), patch("nbskill.knowledge.reference_discover", return_value={}):
        first_install = install_nbskill(**install_kw)
    first = first_install["aai_coding"]
    assert first["installed"]
    skill_text = (aai_root / "skills" / "persistent-python" / "SKILL.md").read_text(encoding="utf-8")
    harness_text = (aai_root / "aai_coding" / "harness.py").read_text(encoding="utf-8")
    assert all(term in skill_text for term in ("nbskill MCP", "healthcheck", "edit_notebook"))
    assert all(term in harness_text for term in ("NBDEV_MSG", "nbskill MCP", "if nbdev:"))
    assert "nbskill-integration:start" in (aai_root / "README.md").read_text(encoding="utf-8")
    assert "nbskill-integration:start" in (aai_root / "SETUP.md").read_text(encoding="utf-8")
    installed_text = {path: path.read_bytes() for path in aai_root.rglob("*") if path.is_file()}
    with patch.object(Path, "home", return_value=aai_root), patch("nbskill.knowledge.reference_discover", return_value={}):
        second_install = install_nbskill(**install_kw)
    assert not second_install["installed"] and second_install["unchanged"]
    assert not second_install["codex_mcp"]["installed"] and not second_install["aai_coding"]["installed"]
    assert installed_text == {path: path.read_bytes() for path in aai_root.rglob("*") if path.is_file()}
finally:
    remove_demo_path(aai_root)